# Project 7bis – Stopword Behaviour in Chat Corpora
## Setup

In [ ]:
# preshed 3.0.14 (released October 2026) breaks spaCy, so we stay on the previous version
%pip install nltk numpy pandas matplotlib scipy certifi jinja2 spacy wordcloud empath scikit-learn gensim torch transformers sentence-transformers accelerate "preshed<3.0.14"

In [ ]:
import json
import re
import ssl
import textwrap
import urllib.request
from collections import Counter
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from nltk.corpus import stopwords
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
FIG_DIR = Path('figures')          # all plots are also saved here
FIG_DIR.mkdir(exist_ok=True)
nltk.download('stopwords', quiet=True)

# The 8 topics of MultiWOZ 2.2 and the colour used for each topic in the plots
TOPICS = ['restaurant', 'hotel', 'train', 'attraction', 'taxi', 'hospital', 'police', 'bus']
TOPIC_COLORS = {
    'restaurant': '#2a78d6',
    'hotel': '#eb6834',
    'train': '#1baf7a',
    'attraction': '#eda100',
    'taxi': '#e87ba4',
    'hospital': '#008300',
    'police': '#6250d6',
    'bus': '#e34948',
}

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/budzianowski/multiwoz/master/data/MultiWOZ_2.2'
DATA_DIR = Path('data/MultiWOZ_2.2')

# The files of MultiWOZ 2.2: the schema, the dialogue acts and the dialogues of the three splits
FILES = ['schema.json', 'dialog_acts.json']
for number in range(1, 18):
    FILES.append(f'train/dialogues_{number:03d}.json')
for split in ['dev', 'test']:
    for number in [1, 2]:
        FILES.append(f'{split}/dialogues_{number:03d}.json')

# certifi avoids SSL errors on some Python installs
try:
    import certifi
    ssl_context = ssl.create_default_context(cafile=certifi.where())
except ImportError:
    ssl_context = ssl.create_default_context()

# Download the files that are not on disk yet
for file_name in FILES:
    target = DATA_DIR / file_name
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(f'{BASE_URL}/{file_name}', context=ssl_context) as response:
            target.write_bytes(response.read())

# Load the dialogue acts and all dialogues
with open(DATA_DIR / 'dialog_acts.json') as file:
    dialog_acts = json.load(file)

dialogues = []
for path in sorted(DATA_DIR.glob('*/dialogues_*.json')):
    with open(path) as file:
        dialogues.extend(json.load(file))

number_of_turns = sum(len(dialogue['turns']) for dialogue in dialogues)
print(f'Dialogues: {len(dialogues):,}')
print(f'Turns    : {number_of_turns:,}')

## Part 1 – Stopwords per Topic
### 1.1 Group the utterances into 8 topics

In [ ]:
rows = []
for dialogue in dialogues:
    previous_topics = set()                  # topics of the previous utterance in this dialogue
    for turn in dialogue['turns']:
        # 1. Topics named in the dialogue acts, e.g. 'Train-Inform' -> 'train'
        acts = dialog_acts[dialogue['dialogue_id']][turn['turn_id']]['dialog_act']
        topics = set()
        for act in acts:
            domain = act.split('-')[0].lower()
            if domain in TOPICS:
                topics.add(domain)

        # 2. For user turns, also the services with an active intent, e.g. 'find_hotel' -> 'hotel'
        if turn['speaker'] == 'USER':
            for frame in turn['frames']:
                if frame['state']['active_intent'] != 'NONE':
                    topics.add(frame['service'])

        # 3. No topic of its own (e.g. "Thank you"): use the topic of the previous utterance
        if not topics:
            topics = previous_topics
        previous_topics = topics

        rows.append({'dialogue_id': dialogue['dialogue_id'], 'speaker': turn['speaker'],
                     'utterance': turn['utterance'], 'topics': sorted(topics)})

turns = pd.DataFrame(rows)

# Leave out the utterances that still have no topic (the opening greetings of a few dialogues)
turns = turns[turns.topics.str.len() > 0].reset_index(drop=True)

# Number of utterances in each topic subgroup
utterances_per_topic = turns.explode('topics').topics.value_counts()
utterances_per_topic.reindex(TOPICS).rename('utterances').to_frame()

### 1.2 Stopwords and their proportion in each subgroup

In [ ]:
STOPWORDS = set(stopwords.words('english'))


def tokenize(text):
    # Lower-case word tokens. Contractions such as "don't" stay whole; numbers and punctuation are dropped.
    text = text.lower().replace('’', "'")             # curly apostrophe -> straight apostrophe
    pieces = re.findall(r"[a-z0-9']+", text)                # runs of letters, digits and apostrophes
    words = []
    for piece in pieces:
        piece = piece.strip("'")
        if re.fullmatch(r"[a-z]+(?:'[a-z]+)*", piece):    # keep only real words
            words.append(piece)
    return words


turns['tokens'] = turns.utterance.map(tokenize)

# Count every word in each topic subgroup (an utterance with two topics counts in both)
token_counts = {topic: Counter() for topic in TOPICS}
for tokens, topics in zip(turns.tokens, turns.topics):
    for topic in topics:
        token_counts[topic].update(tokens)

# The same counts, but only for the stopwords
stop_counts = {}
for topic in TOPICS:
    stop_counts[topic] = Counter({word: count for word, count in token_counts[topic].items() if word in STOPWORDS})

summary = pd.DataFrame({
    'tokens': [sum(token_counts[topic].values()) for topic in TOPICS],
    'stopword_tokens': [sum(stop_counts[topic].values()) for topic in TOPICS],
    'different_stopwords': [len(stop_counts[topic]) for topic in TOPICS],
}, index=TOPICS)
summary['stopword_proportion'] = (summary.stopword_tokens / summary.tokens).round(3)
summary

In [ ]:
# The stopwords found in each subgroup, most frequent first
for topic in TOPICS:
    words = [word for word, count in stop_counts[topic].most_common()]
    print(f'{topic.upper()} ({len(words)} of {len(STOPWORDS)} NLTK stopwords)')
    print(textwrap.fill(', '.join(words), 110), '\n')

### 1.3 Histogram of stopword frequencies

In [ ]:
for topic in TOPICS:
    most_common = stop_counts[topic].most_common()
    words = [word for word, count in most_common]
    counts = [count for word, count in most_common]
    proportion = summary.loc[topic, 'stopword_proportion']

    fig, ax = plt.subplots(figsize=(max(8, 0.14 * len(words) + 2), 4.5))
    ax.bar(range(len(words)), counts, color=TOPIC_COLORS[topic])
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, rotation=90, fontsize=7.5)
    ax.set_xlim(-0.7, len(words) - 0.3)
    ax.set_ylabel('Frequency')
    ax.set_title(f'{topic.capitalize()}: frequency of each stopword '
                 f'(stopwords = {proportion:.1%} of all tokens)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'p1_histogram_{topic}.png', dpi=150, bbox_inches='tight')
    plt.show()

### 1.4 Are some stopwords more frequent in certain topics?

In [ ]:
# Topics used in the tests (bus is too small)
TEST_TOPICS = [topic for topic in TOPICS if topic != 'bus']

# Only utterances with exactly one topic, so that every count belongs to exactly one group
single = turns[turns.topics.str.len() == 1].copy()
single['topic'] = single.topics.str[0]
single = single[single.topic.isin(TEST_TOPICS)]

# Contingency table: how often each stopword occurs in each topic
stopwords_per_topic = {topic: Counter() for topic in TEST_TOPICS}
for tokens, topic in zip(single.tokens, single.topic):
    stopwords_per_topic[topic].update(word for word in tokens if word in STOPWORDS)
table = pd.DataFrame(stopwords_per_topic).fillna(0)

# The chi-square test needs an expected count of at least 5 in every cell
expected_counts = stats.contingency.expected_freq(table)
table = table[(expected_counts >= 5).all(axis=1)]

chi2, p, dof, expected = stats.chi2_contingency(table)
total = table.values.sum()
cramers_v = np.sqrt(chi2 / (total * (min(table.shape) - 1)))

print(f'Stopwords tested: {len(table)}')
p_text = f'= {p:.3g}' if p > 0 else '< 1e-300'               # p is too small to store as a float
print(f'chi2 = {chi2:,.0f}, dof = {dof}, p {p_text}, Cramer\'s V = {cramers_v:.3f}')

In [ ]:
# Adjusted standardised residual of every cell: (observed - expected) divided by its standard error
observed = table.values
total = observed.sum()
stopword_share = observed.sum(axis=1, keepdims=True) / total      # share of each stopword (rows)
topic_share = observed.sum(axis=0, keepdims=True) / total         # share of each topic (columns)
residuals = (observed - expected) / np.sqrt(expected * (1 - stopword_share) * (1 - topic_share))
residuals = pd.DataFrame(residuals, index=table.index, columns=TEST_TOPICS)

# Critical value with a Bonferroni correction for the number of cells
z_crit = stats.norm.ppf(1 - 0.05 / (2 * residuals.size))

# The 25 stopwords that contribute most to the chi-square statistic
contribution = ((observed - expected) ** 2 / expected).sum(axis=1)
top25 = contribution.argsort()[::-1][:25]
top_residuals = residuals.iloc[top25]

fig, ax = plt.subplots(figsize=(9, 11))
image = ax.imshow(top_residuals.clip(-25, 25), cmap='RdBu_r', norm=TwoSlopeNorm(0, -25, 25), aspect='auto')
for row in range(top_residuals.shape[0]):
    for column in range(top_residuals.shape[1]):
        value = top_residuals.iat[row, column]
        label = f'{value:.0f}' + ('•' if abs(value) > z_crit else '')
        ax.text(column, row, label, ha='center', va='center', fontsize=8,
                color='white' if abs(value) > 18 else 'black')
ax.set_xticks(range(len(TEST_TOPICS)))
ax.set_xticklabels([topic.capitalize() for topic in TEST_TOPICS], rotation=30, ha='right')
ax.set_yticks(range(len(top_residuals)))
ax.set_yticklabels(top_residuals.index)
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.6, extend='both', label='Standardised residual')
ax.set_title('Over- (red) and under-represented (blue) stopwords per topic', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p1_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Lift = how many times more frequent a stopword is in a topic than in all topics together
words_per_topic = pd.Series({topic: single.tokens[single.topic == topic].str.len().sum() for topic in TEST_TOPICS})
rate_in_topic = table / words_per_topic                        # occurrences per word, in each topic
rate_overall = table.sum(axis=1) / words_per_topic.sum()       # occurrences per word, all topics together
lift = rate_in_topic.div(rate_overall, axis=0)

# The 5 most over-represented stopwords per topic (largest significant residuals), with their lift
top5 = {}
for topic in TEST_TOPICS:
    significant = residuals[topic][residuals[topic] > z_crit]
    best_words = significant.nlargest(5).index
    top5[topic.capitalize()] = [f'{word} (×{lift.loc[word, topic]:.1f})' for word in best_words]
pd.DataFrame(top5, index=range(1, 6)).T

## Part 2 – How well does the NLTK stopword list fit each topic?

In [ ]:
rows = []
for topic in TOPICS:
    L = set(stop_counts[topic])                                    # the stopwords found in this topic
    most_frequent = [word for word, count in token_counts[topic].most_common(len(L))]   # its |L| most frequent words
    also_in_L = sum(word in L for word in most_frequent)
    rows.append({'topic': topic, '|L|': len(L), 'top words also in L': also_in_L,
                 'overlap': round(also_in_L / len(L), 3)})

overlap = pd.DataFrame(rows).set_index('topic')
overlap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar([topic.capitalize() for topic in TOPICS], overlap.overlap, color=[TOPIC_COLORS[topic] for topic in TOPICS])
for position, value in enumerate(overlap.overlap):
    ax.text(position, value + 0.01, f'{value:.0%}', ha='center', fontsize=10)
ax.set_ylim(0, 1)
ax.set_ylabel('Share of the |L| most frequent words that are in L')
ax.set_title('How well the NLTK stopword list matches the most frequent words', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p2_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for topic in TOPICS:
    if topic == 'bus':
        continue                                       # too few words for a meaningful curve
    L = set(stop_counts[topic])
    most_frequent = [word for word, count in token_counts[topic].most_common(len(L))]
    is_stopword = [word in L for word in most_frequent]          # True if the k-th most frequent word is in L
    k = np.arange(1, len(L) + 1)
    share = np.cumsum(is_stopword) / k                           # share of words in L among the top k words
    ax.plot(k, share, color=TOPIC_COLORS[topic], linewidth=2, label=topic.capitalize())
ax.set_ylim(0, 1.02)
ax.set_xlabel('k (number of most frequent words)')
ax.set_ylabel('Share of the top-k words that are in L')
ax.set_title('Overlap with the NLTK list as more frequent words are included', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'p2_overlap_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for topic in ['restaurant', 'hotel', 'train', 'taxi']:
    L = set(stop_counts[topic])
    most_frequent = [word for word, count in token_counts[topic].most_common(len(L))]
    missing = [word for word in most_frequent if word not in L]              # frequent, but not stopwords
    rare = [word for word, count in stop_counts[topic].most_common()
            if word not in set(most_frequent)]                               # stopwords, but not frequent
    print(f'{topic.upper()}')
    print(textwrap.fill('Frequent but not in the list: ' + ', '.join(missing[:30]), 110))
    print(textwrap.fill('In the list but rare        : ' + ', '.join(rare[-30:]), 110), '\n')

## Part 3 – Named Entities and the Stopwords Next to Them
### 3.1 Find the named entities

In [ ]:
import spacy
from wordcloud import WordCloud

if not spacy.util.is_package('en_core_web_sm'):
    spacy.cli.download('en_core_web_sm')
nlp = spacy.load('en_core_web_sm', disable=['parser', 'lemmatizer'])   # we only need the entity recogniser

entities, near_entity, other_stopwords = [], [], []
for doc in nlp.pipe(turns.utterance, batch_size=1000):
    # Positions of the words directly before and directly after each entity
    next_to_entity = set()
    for entity in doc.ents:
        next_to_entity.add(entity.start - 1)
        next_to_entity.add(entity.end)

    # Split the stopwords (outside entities) into "next to an entity" and "other"
    near, other = [], []
    for position, token in enumerate(doc):
        word = token.lower_
        if word in STOPWORDS and not token.ent_type_:
            if position in next_to_entity:
                near.append(word)
            else:
                other.append(word)

    entities.append([(entity.text.lower(), entity.label_) for entity in doc.ents])
    near_entity.append(near)
    other_stopwords.append(other)

turns['entities'] = entities
turns['near_entity'] = near_entity
turns['other_stopwords'] = other_stopwords

In [ ]:
def utterances_of(topic):
    # All utterances of a topic subgroup (also those that belong to two topics)
    belongs = [topic in topics for topics in turns.topics]
    return turns[belongs]


# Overview: how many entities per topic, and which entity types are most common
rows = []
for topic in TOPICS:
    subgroup = utterances_of(topic)
    topic_entities = []
    for utterance_entities in subgroup.entities:
        topic_entities.extend(utterance_entities)
    type_counts = Counter(label for text, label in topic_entities)
    top3_types = [f'{label} ({count / len(topic_entities):.0%})' for label, count in type_counts.most_common(3)]
    rows.append({'topic': topic, 'utterances': len(subgroup), 'entities': len(topic_entities),
                 'entities per utterance': round(len(topic_entities) / len(subgroup), 2),
                 'most common types': ', '.join(top3_types)})
pd.DataFrame(rows).set_index('topic')

In [ ]:
# How often each (entity, type) pair was found in the whole corpus
entity_counts = Counter()
for utterance_entities in turns.entities:
    entity_counts.update(utterance_entities)

# For every entity type: the number of entities and its 6 most common examples
type_counts, type_examples = Counter(), {}
for (text, label), count in entity_counts.most_common():
    type_counts[label] += count
    type_examples.setdefault(label, [])
    if len(type_examples[label]) < 6:
        type_examples[label].append(text)

entity_types = pd.DataFrame({
    'meaning': {label: spacy.explain(label) for label in type_counts},
    'entities': type_counts,
    'most common examples': {label: ', '.join(examples) for label, examples in type_examples.items()},
}).sort_values('entities', ascending=False)

with pd.option_context('display.max_colwidth', None):
    display(entity_types)

### 3.2 Word clouds of the named entities

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 18))
for ax, topic in zip(axes.flat, TOPICS):
    # How often each entity occurs in the topic
    frequencies = Counter()
    for utterance_entities in utterances_of(topic).entities:
        frequencies.update(text for text, label in utterance_entities)

    color = TOPIC_COLORS[topic]
    cloud = WordCloud(width=800, height=400, background_color='white', random_state=42,
                      color_func=lambda *args, **kwargs: color)        # every word in the topic's colour
    ax.imshow(cloud.generate_from_frequencies(frequencies))
    ax.set_title(f'{topic.capitalize()} ({sum(frequencies.values()):,} entities)', fontweight='bold', fontsize=13)
    ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p3_wordclouds.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.3 Stopwords right next to a named entity

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 20))
for ax, topic in zip(axes.flat, TOPICS):
    near_counts = Counter()
    for words in utterances_of(topic).near_entity:
        near_counts.update(words)
    top15 = near_counts.most_common(15)[::-1]          # reversed, so that the most frequent is at the top
    ax.barh([word for word, count in top15], [count for word, count in top15], color=TOPIC_COLORS[topic])
    ax.set_title(f'{topic.capitalize()}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Frequency next to an entity')
    ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p3_stopwords_near_entities.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Testing the trends

In [ ]:
# Single-topic utterances without bus, as in Part 1
single = turns[turns.topics.str.len() == 1].copy()
single['topic'] = single.topics.str[0]
single = single[single.topic.isin(TEST_TOPICS)]

# Test 1: how often each stopword occurs next to an entity, per topic
near_counts = {}
for topic in TEST_TOPICS:
    near_counts[topic] = Counter()
    for words in single.near_entity[single.topic == topic]:
        near_counts[topic].update(words)
near_table = pd.DataFrame(near_counts).fillna(0)

# Keep only the stopwords with an expected count of at least 5 in every topic
expected_counts = stats.contingency.expected_freq(near_table)
near_table = near_table[(expected_counts >= 5).all(axis=1)]

chi2, p, dof, expected = stats.chi2_contingency(near_table)
cramers_v = np.sqrt(chi2 / (near_table.values.sum() * (min(near_table.shape) - 1)))
p_text = f'= {p:.3g}' if p > 0 else '< 1e-300'
print(f'Stopwords tested: {len(near_table)}')
print(f'chi2 = {chi2:,.0f}, dof = {dof}, p {p_text}, Cramer\'s V = {cramers_v:.3f}')

# The 3 stopwords most over-represented next to entities in each topic (largest standardised residual)
near_residuals = (near_table - expected) / np.sqrt(expected)
top3 = {topic.capitalize(): near_residuals[topic].nlargest(3).index.tolist() for topic in TEST_TOPICS}
pd.DataFrame(top3, index=['1st', '2nd', '3rd']).T

In [ ]:
rows = []
for topic in TEST_TOPICS:
    subgroup = single[single.topic == topic]

    # Count the stopwords next to entities and all other stopwords
    near, other = Counter(), Counter()
    for words in subgroup.near_entity:
        near.update(words)
    for words in subgroup.other_stopwords:
        other.update(words)

    # Test 2: chi-square test of "next to entities" against "elsewhere"
    topic_table = pd.DataFrame({'near': near, 'other': other}).fillna(0)
    expected_counts = stats.contingency.expected_freq(topic_table)
    topic_table = topic_table[(expected_counts >= 5).all(axis=1)]
    chi2, p, dof, _ = stats.chi2_contingency(topic_table)
    cramers_v = np.sqrt(chi2 / topic_table.values.sum())

    # Enrichment = share of a stopword next to entities / its share elsewhere
    # (only stopwords seen at least 20 times next to an entity)
    share = topic_table / topic_table.sum()
    enrichment = share.near / share.other
    enrichment = enrichment[topic_table.near >= 20].nlargest(5)

    rows.append({'topic': topic, 'chi2': round(chi2), 'p': f'{p:.2g}' if p > 0 else '< 1e-300',
                 "Cramer's V": round(cramers_v, 3),
                 'most enriched next to entities': ', '.join(f'{word} (×{times:.0f})'
                                                            for word, times in enrichment.items())})
pd.DataFrame(rows).set_index('topic')

## Part 4 – Vocabulary Growth and Heaps' Law
### 4.1 Vocabulary growth

In [ ]:
growth = {}
for topic in TOPICS:
    # All tokens of the topic, in corpus order
    tokens = []
    for utterance_tokens, topics in zip(turns.tokens, turns.topics):
        if topic in topics:
            tokens.extend(utterance_tokens)

    # Vocabulary size after each token
    seen = set()
    vocabulary_size = []
    for word in tokens:
        seen.add(word)
        vocabulary_size.append(len(seen))

    # Keep 200 checkpoints, spread evenly on a log scale
    N = np.unique(np.geomspace(1, len(tokens), 200).astype(int))      # number of tokens read
    V = np.array(vocabulary_size)[N - 1]                              # vocabulary size at that point
    growth[topic] = (N, V)

fig, ax = plt.subplots(figsize=(10, 5.5))
for topic in TOPICS:
    N, V = growth[topic]
    ax.plot(N, V, color=TOPIC_COLORS[topic], linewidth=2, label=topic.capitalize())
ax.set_xlabel('Number of tokens N')
ax.set_ylabel('Vocabulary size V')
ax.set_title('Vocabulary growth in each topic', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_vocabulary_growth.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Fitting Heaps' law

In [ ]:
def r2(actual, predicted):
    # Coefficient of determination R²
    return 1 - np.sum((actual - predicted) ** 2) / np.sum((actual - actual.mean()) ** 2)


def fit_heaps(N, V):
    # Fit V = K * N^beta as a straight line on the log values: log V = log K + beta * log N
    beta, log_K = np.polyfit(np.log(N), np.log(V), 1)
    K = np.exp(log_K)
    V_fit = K * N ** beta
    return {'K': K, 'beta': beta,
            'R2 (log-log)': r2(np.log(V), np.log(V_fit)),
            'R2 (original scale)': r2(V, V_fit),
            'mean error (%)': 100 * np.mean(np.abs(V_fit - V) / V)}


fits_all, fits_tail = {}, {}
for topic in TOPICS:
    N, V = growth[topic]
    fits_all[topic] = fit_heaps(N, V)                        # fit on the whole curve
    if N[-1] >= 10_000:                                      # bus has too few tokens for the second fit
        long_run = N >= 1000
        fits_tail[topic] = fit_heaps(N[long_run], V[long_run])   # fit after the first 1,000 tokens

print('Fit on all N:')
display(pd.DataFrame(fits_all).T.round(3))
print('Fit on N >= 1,000 (long-run growth):')
display(pd.DataFrame(fits_tail).T.round(3))

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 17))
for ax, topic in zip(axes.flat, TOPICS):
    N, V = growth[topic]
    ax.loglog(N, V, color=TOPIC_COLORS[topic], linewidth=3, label='actual vocabulary')

    fit = fits_all[topic]
    ax.loglog(N, fit['K'] * N ** fit['beta'], color='black', linestyle='--', linewidth=1.2,
              label=f"all N: β = {fit['beta']:.2f}, R² = {fit['R2 (log-log)']:.3f}")

    if topic in fits_tail:
        fit = fits_tail[topic]
        N_long_run = N[N >= 1000]
        ax.loglog(N_long_run, fit['K'] * N_long_run ** fit['beta'], color='black', linestyle=':', linewidth=2,
                  label=f"N ≥ 1,000: β = {fit['beta']:.2f}, R² = {fit['R2 (log-log)']:.3f}")

    ax.set_title(topic.capitalize(), fontweight='bold')
    ax.set_xlabel('Number of tokens N')
    ax.set_ylabel('Vocabulary size V')
    ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_heaps_fits.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
new_words = {}
for topic in TOPICS:
    N, V = growth[topic]
    start = int(0.9 * N[-1])                          # first token of the last 10%
    vocabulary_at_start = np.interp(start, N, V)      # vocabulary size at that point
    new_words[topic] = round(1000 * (V[-1] - vocabulary_at_start) / (N[-1] - start), 1)
pd.Series(new_words, name='new words per 1,000 tokens (last 10%)').to_frame()

## Part 5 – Semantic Content of the Topics (Empath)
### 5.1 The 5 dominant categories of each topic

In [ ]:
from empath import Empath

lexicon = Empath()

# Empath splits text on spaces only, so we give it our cleaned, lower-case tokens from Part 1
empath_scores = {}
for topic in TOPICS:
    topic_words = []
    for tokens, topics in zip(turns.tokens, turns.topics):
        if topic in topics:
            topic_words.extend(tokens)
    empath_scores[topic] = lexicon.analyze(' '.join(topic_words), normalize=True)

# Rows: the 194 categories; columns: the topics; values: occurrences per 1,000 words
empath_scores = pd.DataFrame(empath_scores) * 1000

top5 = {}
for topic in TOPICS:
    best_categories = empath_scores[topic].nlargest(5).index
    top5[topic.capitalize()] = [f'{category} ({empath_scores.loc[category, topic]:.0f})' for category in best_categories]
print('Top 5 Empath categories per topic (occurrences per 1,000 words):')
pd.DataFrame(top5, index=range(1, 6)).T

In [ ]:
for topic in TOPICS:
    word_counts = token_counts[topic]                          # word counts of the topic, from Part 1
    print(topic.upper())
    for category in empath_scores[topic].nlargest(5).index:
        # The category's words that occur in the topic, most frequent first
        category_words = set(lexicon.cats[category]) & set(word_counts)
        most_frequent = sorted(category_words, key=lambda word: (-word_counts[word], word))[:6]   # ties: alphabetical
        print(f'  {category:<18} ' + ', '.join(most_frequent))
    print()

In [ ]:
from matplotlib.patches import Rectangle

# The categories that are in the top 5 of at least one topic, in order of appearance
categories = []
for topic in TOPICS:
    for category in empath_scores[topic].nlargest(5).index:
        if category not in categories:
            categories.append(category)
values = empath_scores.loc[categories, TOPICS]

fig, ax = plt.subplots(figsize=(10, 0.45 * len(categories) + 2))
image = ax.imshow(values.values, cmap='Blues', aspect='auto')
for row, category in enumerate(categories):
    for column, topic in enumerate(TOPICS):
        value = values.iat[row, column]
        ax.text(column, row, f'{value:.0f}', ha='center', va='center', fontsize=8,
                color='white' if value > 0.6 * values.values.max() else 'black')
        if category in empath_scores[topic].nlargest(5).index:          # frame the topic's own top 5
            ax.add_patch(Rectangle((column - 0.5, row - 0.5), 1, 1, fill=False, edgecolor='black', linewidth=2))
ax.set_xticks(range(len(TOPICS)))
ax.set_xticklabels([topic.capitalize() for topic in TOPICS], rotation=30, ha='right')
ax.set_yticks(range(len(categories)))
ax.set_yticklabels(categories)
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.6, label='Occurrences per 1,000 words')
ax.set_title('Dominant Empath categories per topic (framed = top 5 of that topic)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p5_empath_top5.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Similarity between the topics

In [ ]:
def cosine_matrix(profiles):
    # Cosine similarity between every pair of columns (topics)
    unit_vectors = profiles / np.linalg.norm(profiles, axis=0)
    return pd.DataFrame(unit_vectors.T @ unit_vectors, index=profiles.columns, columns=profiles.columns)


raw_sim = cosine_matrix(empath_scores[TOPICS])

# Centred profiles: subtract the average topic (without bus) from every topic
average_profile = empath_scores[TEST_TOPICS].mean(axis=1)
centred_sim = cosine_matrix(empath_scores[TOPICS].sub(average_profile, axis=0))

panels = [('Raw Empath profiles', raw_sim, 'Blues', None),
          ('Centred Empath profiles', centred_sim, 'RdBu_r', TwoSlopeNorm(0, -1, 1))]
labels = [topic.capitalize() for topic in TOPICS]

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
for ax, (title, similarity, cmap, norm) in zip(axes, panels):
    image = ax.imshow(similarity.values, cmap=cmap, norm=norm)
    for row in range(len(TOPICS)):
        for column in range(len(TOPICS)):
            value = similarity.iat[row, column]
            ax.text(column, row, f'{value:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(value) > 0.75 else 'black')
    ax.set_xticks(range(len(TOPICS)))
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_yticks(range(len(TOPICS)))
    ax.set_yticklabels(labels)
    ax.grid(False)
    ax.set_title(f'Cosine similarity – {title}', fontweight='bold')
    fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p5_cosine_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Every pair of topics (bus left out: too small), from most to least similar
pairs = []
for first, topic_1 in enumerate(TEST_TOPICS):
    for topic_2 in TEST_TOPICS[first + 1:]:
        pairs.append((topic_1, topic_2, raw_sim.loc[topic_1, topic_2], centred_sim.loc[topic_1, topic_2]))
pairs = pd.DataFrame(pairs, columns=['topic 1', 'topic 2', 'raw cosine', 'centred cosine'])
pairs.sort_values('centred cosine', ascending=False).round(3).reset_index(drop=True)

## Part 6 – Predicting the Intent with Empath

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

with open(DATA_DIR / 'schema.json') as file:
    schema = json.load(file)

# Two intents have different names in schema.json and in the dialogue annotations; we use the dialogue names
RENAMED = {'book_taxi': 'find_taxi', 'police': 'find_police'}

# INTENTS[topic][intent name] = description of the intent
INTENTS = {}
for service in schema:
    topic = service['service_name']
    INTENTS[topic] = {}
    for intent in service['intents']:
        name = RENAMED.get(intent['name'], intent['name'])
        INTENTS[topic][name] = intent['description']

for topic in TOPICS:
    descriptions = [f'{name}: "{description}"' for name, description in INTENTS[topic].items()]
    print(f'{topic:<11}', ' | '.join(descriptions))

# User utterances with their annotated intent (one row per topic that is active in the turn)
rows = []
for dialogue in dialogues:
    for turn in dialogue['turns']:
        if turn['speaker'] == 'USER':
            for frame in turn['frames']:
                if frame['state']['active_intent'] != 'NONE':
                    rows.append({'topic': frame['service'], 'utterance': turn['utterance'],
                                 'intent': frame['state']['active_intent']})
labelled = pd.DataFrame(rows)
labelled['tokens'] = labelled.utterance.map(tokenize)
print(f'\nLabelled user utterances: {len(labelled):,}')
labelled.groupby(['topic', 'intent']).size().rename('utterances').to_frame()

### 6.1 Empath vectors

In [ ]:
# Index from every word to the Empath categories it belongs to. This gives the same counts as
# lexicon.analyze, but is much faster for many short utterances.
CATEGORIES = sorted(lexicon.cats)
word_to_categories = {}
for position, category in enumerate(CATEGORIES):
    for word in lexicon.cats[category]:
        word_to_categories.setdefault(word, []).append(position)


def empath_vector(tokens):
    # Number of the utterance's words in each of the 194 categories
    vector = np.zeros(len(CATEGORIES))
    for word in tokens:
        for position in word_to_categories.get(word, []):
            vector[position] += 1
    return vector


# All 11 intents, and the Empath vector of each intent's name + description
ALL_INTENTS = []
intent_vectors = {}
for topic in TOPICS:
    for name, description in INTENTS[topic].items():
        ALL_INTENTS.append(name)
        intent_text = name.replace('_', ' ') + ' ' + description          # e.g. "book restaurant book a table ..."
        intent_vectors[name] = empath_vector(tokenize(intent_text))

# The Empath vector of every labelled utterance (one row per utterance)
empath_matrix = np.vstack(labelled.tokens.map(empath_vector))

print(f'Utterances without any Empath word: {(empath_matrix.sum(axis=1) == 0).mean():.1%}')
print('\nEmpath categories of each intent:')
for name, vector in intent_vectors.items():
    print(f'  {name:<16}', ', '.join(CATEGORIES[position] for position in np.flatnonzero(vector)))

In [ ]:
def predict(vectors, candidates):
    # Assign each utterance vector to the candidate intent with the highest cosine similarity.
    # Utterances without any Empath word, and ties, give 'no decision'.
    intent_matrix = np.vstack([intent_vectors[name] for name in candidates])

    # Cosine similarity = dot product / both vector lengths (a length of 0 is replaced by 1 to avoid dividing by 0)
    utterance_lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    intent_lengths = np.linalg.norm(intent_matrix, axis=1)
    similarity = ((vectors @ intent_matrix.T)
                  / np.where(utterance_lengths == 0, 1, utterance_lengths)
                  / np.where(intent_lengths == 0, 1, intent_lengths))

    best = similarity.max(axis=1, keepdims=True)
    has_signal = best[:, 0] > 0                              # at least one category in common
    no_tie = (similarity == best).sum(axis=1) == 1           # exactly one intent has the best score
    best_intent = np.array(candidates)[similarity.argmax(axis=1)]
    return np.where(has_signal & no_tie, best_intent, 'no decision')


# A: within the topic
labelled['pred_within'] = ''
for topic in TOPICS:
    in_topic = (labelled.topic == topic).to_numpy()
    labelled.loc[in_topic, 'pred_within'] = predict(empath_matrix[in_topic], list(INTENTS[topic]))

# B: among all 11 intents
labelled['pred_all'] = predict(empath_matrix, ALL_INTENTS)

### 6.2 F1-score per topic

In [ ]:
rows = []
for topic in TOPICS:
    subgroup = labelled[labelled.topic == topic]
    topic_intents = list(INTENTS[topic])
    majority = [subgroup.intent.mode()[0]] * len(subgroup)        # baseline: always guess the most common intent

    def topic_macro_f1(predicted):
        return f1_score(subgroup.intent, predicted, labels=topic_intents, average='macro', zero_division=0)

    rows.append({
        'topic': topic, 'intents': len(topic_intents), 'utterances': len(subgroup),
        'baseline accuracy': accuracy_score(subgroup.intent, majority),
        'baseline macro-F1': topic_macro_f1(majority),
        'A accuracy': accuracy_score(subgroup.intent, subgroup.pred_within),
        'A macro-F1': topic_macro_f1(subgroup.pred_within),
        'A no decision': (subgroup.pred_within == 'no decision').mean(),
        'B accuracy': accuracy_score(subgroup.intent, subgroup.pred_all),
        'B macro-F1': topic_macro_f1(subgroup.pred_all),
        'B no decision': (subgroup.pred_all == 'no decision').mean(),
    })
empath_results = pd.DataFrame(rows).set_index('topic')

overall_B = f1_score(labelled.intent, labelled.pred_all, labels=ALL_INTENTS, average='macro', zero_division=0)
print(f'B – macro-F1 over all 11 intents: {overall_B:.3f}')
with pd.option_context('display.max_columns', None, 'display.width', 250):
    display(empath_results.round(3))

In [ ]:
# Share (%) of each true intent's utterances that went to each predicted intent
confusion = pd.crosstab(labelled.intent, labelled.pred_all, normalize='index')
confusion = confusion.reindex(index=ALL_INTENTS, columns=ALL_INTENTS + ['no decision'], fill_value=0) * 100

fig, ax = plt.subplots(figsize=(12, 8))
image = ax.imshow(confusion.values, cmap='Blues', vmin=0, vmax=100, aspect='auto')
for row in range(confusion.shape[0]):
    for column in range(confusion.shape[1]):
        value = confusion.iat[row, column]
        ax.text(column, row, f'{value:.0f}', ha='center', va='center', fontsize=8,
                color='white' if value > 60 else 'black')
ax.set_xticks(range(confusion.shape[1]))
ax.set_xticklabels(confusion.columns, rotation=45, ha='right')
ax.set_yticks(range(confusion.shape[0]))
ax.set_yticklabels(confusion.index)
ax.set_xlabel('Predicted intent')
ax.set_ylabel('True intent')
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.8, label='% of the true intent')
ax.set_title('Empath intent prediction among all 11 intents (B)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p6_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 7 – Predicting the Intent with Word and Sentence Embeddings

In [ ]:
from gensim.models import Word2Vec, FastText
import torch
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging as transformers_logging
from sentence_transformers import SentenceTransformer

transformers_logging.set_verbosity_error()          # hide model-loading reports
transformers_logging.disable_progress_bar()

# Run the transformer models on the GPU if there is one
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'                                  # Apple GPU
else:
    device = 'cpu'
print('Running the transformer models on:', device)

# Every distinct labelled utterance is embedded once; row_of gives each labelled row its position in this list
utterances = labelled.utterance.unique().tolist()
position_of = {utterance: position for position, utterance in enumerate(utterances)}
row_of = labelled.utterance.map(position_of).to_numpy()

# The text of each intent, as in Part 6: name + description
intent_texts = []
for topic in TOPICS:
    for name in INTENTS[topic]:
        intent_texts.append(name.replace('_', ' ') + ' ' + INTENTS[topic][name])

### 7.1 Building the four embeddings

In [ ]:
import zlib


def stable_hash(text):
    # gensim normally uses Python's hash(), which changes on every run; a fixed hash makes the results reproducible
    return zlib.crc32(text.encode())


# word2vec and FastText, trained on all MultiWOZ utterances (user and system).
# gensim may print "Exception ignored in ... our_dot_float": this is harmless, gensim just picks another maths routine.
settings = dict(vector_size=100, window=5, min_count=2, workers=1, seed=42, epochs=10, hashfxn=stable_hash)
w2v = Word2Vec(turns.tokens.tolist(), **settings)
fasttext = FastText(turns.tokens.tolist(), **settings)


def average_word_vectors(model, texts):
    # Average vector of the non-stopword words of each text (zero vector if no word is known)
    vectors = []
    for text in texts:
        words = []
        for word in tokenize(text):
            known = model is fasttext or word in model.wv       # FastText can build a vector for any word
            if word not in STOPWORDS and known:
                words.append(word)
        if words:
            vectors.append(np.mean([model.wv[word] for word in words], axis=0))
        else:
            vectors.append(np.zeros(model.vector_size))
    return np.vstack(vectors)


# RoBERTa: average of the last-layer vectors of all tokens in the sentence
roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
roberta = AutoModel.from_pretrained('roberta-base').to(device).eval()


def roberta_vectors(texts, batch_size=128):
    vectors = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = roberta_tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                                      max_length=64, return_tensors='pt').to(device)
            hidden = roberta(**batch).last_hidden_state              # one vector per token
            mask = batch['attention_mask'].unsqueeze(-1)            # 1 for real tokens, 0 for padding
            sentence_vectors = (hidden * mask).sum(1) / mask.sum(1)  # average over the real tokens
            vectors.append(sentence_vectors.cpu().numpy())
    return np.vstack(vectors)


# Sentence-Transformer: the model returns a sentence vector directly
sentence_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)


def word2vec_vectors(texts):
    return average_word_vectors(w2v, texts)


def fasttext_vectors(texts):
    return average_word_vectors(fasttext, texts)


def sentence_transformer_vectors(texts):
    return sentence_model.encode(texts, batch_size=256)


EMBEDDERS = {
    'word2vec': word2vec_vectors,
    'FastText': fasttext_vectors,
    'RoBERTa': roberta_vectors,
    'Sentence-Transformer': sentence_transformer_vectors,
}

utterance_vectors, intent_vectors_by_model = {}, {}
for name, embed in EMBEDDERS.items():
    utterance_vectors[name] = embed(utterances)[row_of]          # one row per labelled utterance
    intent_vectors_by_model[name] = dict(zip(ALL_INTENTS, embed(intent_texts)))
    print(f'{name:<21} vector size {utterance_vectors[name].shape[1]}')

### 7.2 Predicting and scoring

In [ ]:
def predict_with(vectors, intent_vectors_of_model, candidates):
    # The candidate intent with the highest cosine similarity ('no decision' for zero vectors)
    intent_matrix = np.vstack([intent_vectors_of_model[name] for name in candidates])
    utterance_lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    similarity = ((vectors @ intent_matrix.T)
                  / np.where(utterance_lengths == 0, 1, utterance_lengths)
                  / np.linalg.norm(intent_matrix, axis=1))
    best_intent = np.array(candidates)[similarity.argmax(axis=1)]
    return np.where(utterance_lengths[:, 0] > 0, best_intent, 'no decision')


scores_A, scores_B = {}, {}
for name in EMBEDDERS:
    vectors = utterance_vectors[name]
    intent_vectors_of_model = intent_vectors_by_model[name]

    # A: within the topic
    pred_within = np.empty(len(labelled), dtype=object)
    for topic in TOPICS:
        in_topic = (labelled.topic == topic).to_numpy()
        pred_within[in_topic] = predict_with(vectors[in_topic], intent_vectors_of_model, list(INTENTS[topic]))

    # B: among all 11 intents
    pred_all = predict_with(vectors, intent_vectors_of_model, ALL_INTENTS)

    # Macro-F1 per topic, and over all intents
    scores_A[name], scores_B[name] = {}, {}
    for topic in TOPICS:
        in_topic = (labelled.topic == topic).to_numpy()
        topic_intents = list(INTENTS[topic])
        scores_A[name][topic] = f1_score(labelled.intent[in_topic], pred_within[in_topic], labels=topic_intents,
                                         average='macro', zero_division=0)
        scores_B[name][topic] = f1_score(labelled.intent[in_topic], pred_all[in_topic], labels=topic_intents,
                                         average='macro', zero_division=0)
    scores_B[name]['all 11 intents'] = f1_score(labelled.intent, pred_all, labels=ALL_INTENTS,
                                                average='macro', zero_division=0)

### 7.3 Results

In [ ]:
TWO_INTENT_TOPICS = ['restaurant', 'hotel', 'train']
table_A = pd.DataFrame(scores_A).loc[TWO_INTENT_TOPICS]
table_A.insert(0, 'Empath', empath_results.loc[TWO_INTENT_TOPICS, 'A macro-F1'])
table_A.insert(0, 'baseline', empath_results.loc[TWO_INTENT_TOPICS, 'baseline macro-F1'])
table_A.round(3)

In [ ]:
table_B = pd.DataFrame(scores_B)
table_B.insert(0, 'Empath', list(empath_results['B macro-F1']) + [overall_B])
table_B.round(3)

In [ ]:
MODEL_COLORS = {'Empath': '#9e9e9e', 'word2vec': '#2a78d6', 'FastText': '#1baf7a',
                'RoBERTa': '#eda100', 'Sentence-Transformer': '#6250d6'}
plot_rows = [row for row in table_B.index if row != 'bus']           # bus: only 8 utterances
x = np.arange(len(plot_rows))
width = 0.16

fig, ax = plt.subplots(figsize=(14, 5.5))
for k, model in enumerate(MODEL_COLORS):
    ax.bar(x + (k - 2) * width, table_B.loc[plot_rows, model], width, color=MODEL_COLORS[model], label=model)
ax.set_xticks(x)
ax.set_xticklabels(['ALL 11 INTENTS' if row == 'all 11 intents' else row.capitalize() for row in plot_rows])
ax.set_ylim(0, 1)
ax.set_ylabel('Macro-F1')
ax.set_title('Intent prediction among all 11 intents (B): macro-F1 per topic and model', fontweight='bold')
ax.legend(ncol=5, loc='upper left')
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p7_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 8 – A Trained Classifier for the Intent

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline

# The dialogue of each labelled utterance (same loop and order as in Part 6)
dialogue_ids = []
for dialogue in dialogues:
    for turn in dialogue['turns']:
        if turn['speaker'] == 'USER':
            for frame in turn['frames']:
                if frame['state']['active_intent'] != 'NONE':
                    dialogue_ids.append(dialogue['dialogue_id'])
labelled['dialogue_id'] = dialogue_ids

# 80/20 split by dialogue: every dialogue is entirely in training or entirely in test
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(labelled, groups=labelled.dialogue_id))
train, test = labelled.iloc[train_idx], labelled.iloc[test_idx]
print(f'Training: {len(train):,} utterances from {train.dialogue_id.nunique():,} dialogues')
print(f'Test    : {len(test):,} utterances from {test.dialogue_id.nunique():,} dialogues')

# TF-IDF of words and word pairs (stopwords kept), followed by logistic regression
classifier = make_pipeline(
    TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, ngram_range=(1, 2), min_df=2,
                    sublinear_tf=True),
    LogisticRegression(max_iter=2000))
classifier.fit(train.utterance, train.intent)
print(f"Model trained: {len(classifier[0].vocabulary_):,} words and word pairs")

### 8.1 Predictions and F1-scores

In [ ]:
classes = list(classifier.classes_)                          # the intents, in the classifier's order
probabilities = classifier.predict_proba(test.utterance)     # one probability per intent and utterance

# B: the most probable of all intents
test = test.assign(pred_all=np.array(classes)[probabilities.argmax(axis=1)])

# A: the most probable of the topic's own intents
pred_within = np.empty(len(test), dtype=object)
for topic in TOPICS:
    in_topic = (test.topic == topic).to_numpy()
    own_columns = [classes.index(name) for name in INTENTS[topic]]
    best_own = probabilities[in_topic][:, own_columns].argmax(axis=1)
    pred_within[in_topic] = np.array(classes)[own_columns][best_own]
test = test.assign(pred_within=pred_within)

# Sentence-Transformer (Part 7) on the same test utterances
st_pred = predict_with(utterance_vectors['Sentence-Transformer'][test_idx],
                       intent_vectors_by_model['Sentence-Transformer'], ALL_INTENTS)

rows = []
for topic in TOPICS:
    in_topic = (test.topic == topic).to_numpy()
    topic_intents = list(INTENTS[topic])

    def topic_macro_f1(predicted):
        return f1_score(test.intent[in_topic], predicted, labels=topic_intents, average='macro', zero_division=0)

    has_test_data = in_topic.any()
    rows.append({'topic': topic, 'test utterances': in_topic.sum(),
                 'A macro-F1 (classifier)': topic_macro_f1(test.pred_within[in_topic]) if len(topic_intents) > 1 else np.nan,
                 'B macro-F1 (classifier)': topic_macro_f1(test.pred_all[in_topic]) if has_test_data else np.nan,
                 'B macro-F1 (Sentence-Transformer, Part 7)': topic_macro_f1(st_pred[in_topic]) if has_test_data else np.nan})
ml_results = pd.DataFrame(rows).set_index('topic')

# Overall macro-F1 over the intents that occur in the test set (the few bus dialogues may all be in training)
test_intents = [name for name in ALL_INTENTS if name in set(test.intent)]
ml_results.loc['all intents'] = [len(test), np.nan,
                                 f1_score(test.intent, test.pred_all, labels=test_intents, average='macro'),
                                 f1_score(test.intent, st_pred, labels=test_intents, average='macro')]
ml_results.round(3)

In [ ]:
print(classification_report(test.intent, test.pred_all, labels=test_intents, digits=3, zero_division=0))

In [ ]:
# The 8 words and word pairs with the largest weight for each intent
features = classifier[0].get_feature_names_out()
weights = classifier[1].coef_                        # one row of weights per intent
for position, intent in enumerate(classes):
    strongest = np.argsort(weights[position])[::-1][:8]
    print(f'{intent:<16}', ', '.join(features[strongest]))

## Part 9 – Predicting the Intent with an LLM (Phi-3.5-mini-instruct)
### 9.1 Inspiration from existing MultiWOZ work
### 9.2 A fair test sample

In [ ]:
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, DynamicCache

# The previous 3 turns of each labelled utterance (same loop and order as in Part 6)
contexts = []
for dialogue in dialogues:
    for position, turn in enumerate(dialogue['turns']):
        if turn['speaker'] != 'USER':
            continue
        previous_turns = dialogue['turns'][max(0, position - 3):position]
        context = [(previous['speaker'].lower(), previous['utterance']) for previous in previous_turns]
        for frame in turn['frames']:
            if frame['state']['active_intent'] != 'NONE':
                contexts.append(context)
labelled['context'] = contexts

# The official MultiWOZ split (train / dev / test) of every dialogue
official_split = {}
for path in DATA_DIR.glob('*/dialogues_*.json'):
    with open(path) as file:
        for dialogue in json.load(file):
            official_split[dialogue['dialogue_id']] = path.parent.name

# Test sample: Part 8 test utterances from the official dev/test dialogues, in the five standard topics
FIVE_TOPICS = ['restaurant', 'hotel', 'train', 'attraction', 'taxi']
pool = labelled.iloc[test_idx]
unseen = pool.dialogue_id.map(official_split).isin(['dev', 'test'])
pool = pool[unseen & pool.topic.isin(FIVE_TOPICS)]
sample = pool.sample(1000, random_state=42)
print(f'Unseen test utterances: {len(pool):,}; random sample: {len(sample):,}')
sample.groupby(['topic', 'intent']).size().rename('utterances').to_frame()

### 9.3 The prompt

In [ ]:
# The 11 intents become the answer options A–K
LETTERS = 'ABCDEFGHIJK'
LETTER_OF = dict(zip(ALL_INTENTS, LETTERS))
option_lines = []
for topic in TOPICS:
    for name in INTENTS[topic]:
        option_lines.append(f'{LETTER_OF[name]}. {name} – {INTENTS[topic][name]}')
OPTIONS = '\n'.join(option_lines)

SYSTEM = ('You are an expert annotator of the MultiWOZ dataset: written conversations between a tourist (user) '
          'and a clerk (system) of the Cambridge TownInfo centre.')
RULES = ("- Use find_... while the user is searching or asking for information (area, price, food, address, phone).\n"
         "- Use book_... when the user wants to reserve, gives booking details (day, time, number of people, nights) "
         "or confirms a booking.\n"
         "- Short replies such as 'yes please' or 'thank you' belong to the intent that is being discussed.")

# One example per intent from the training set (an utterance of at least 6 words, chosen at random)
rng_examples = np.random.default_rng(0)
examples = []
for name in ALL_INTENTS:
    candidates = train[(train.intent == name) & (train.utterance.str.split().str.len() >= 6)]
    chosen = candidates.index[rng_examples.integers(len(candidates))]
    examples.append(labelled.loc[chosen])


def conversation(row, with_context):
    # The utterance as a conversation: optionally the previous turns, then "user: <utterance>"
    lines = []
    if with_context:
        lines = [f'{speaker}: {text}' for speaker, text in row.context]
    lines.append(f'user: {row.utterance}')
    return '\n'.join(lines)


def build_prompt(row, with_context):
    shot_texts = []
    for example in examples:
        shot_texts.append(f'Conversation:\n{conversation(example, with_context)}\nAnswer: {LETTER_OF[example.intent]}')
    shots = '\n\n'.join(shot_texts)
    user_message = ('Which intent is the user pursuing in their last message? Choose exactly one option.\n\n'
                    f'Options:\n{OPTIONS}\n\nRules:\n{RULES}\n\nExamples:\n\n{shots}\n\n'
                    f'Now classify this conversation:\n{conversation(row, with_context)}\n\nAnswer with the letter only.')
    return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user_message}]


print(build_prompt(sample.iloc[0], with_context=True)[1]['content'])

### 9.4 Running Phi-3.5

In [ ]:
phi_tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3.5-mini-instruct')
phi = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3.5-mini-instruct', dtype=torch.bfloat16).to(device).eval()

# Token ids of the answer letters (with and without a leading space)
letter_ids = []
for letter in LETTERS:
    candidates = {phi_tokenizer.convert_tokens_to_ids(letter), phi_tokenizer.convert_tokens_to_ids('▁' + letter)}
    letter_ids.append([token_id for token_id in candidates if token_id not in (None, phi_tokenizer.unk_token_id)])

MARK = 'Now classify this conversation:\n'          # everything before this is the same for every utterance


def phi_letter_scores(df, with_context):
    # Score of every answer letter for every utterance (columns in the order of ALL_INTENTS)

    def prompt_as_text(row):
        return phi_tokenizer.apply_chat_template(build_prompt(row, with_context),
                                                 add_generation_prompt=True, tokenize=False)

    # The shared beginning of the prompt is processed once and kept in the cache
    first_prompt = prompt_as_text(df.iloc[0])
    prefix = first_prompt[:first_prompt.index(MARK) + len(MARK)]
    prefix_ids = phi_tokenizer(prefix, return_tensors='pt', add_special_tokens=False)['input_ids'].to(device)
    cache = DynamicCache()

    scores = []
    with torch.no_grad():
        phi(prefix_ids, past_key_values=cache)
        for _, row in df.iterrows():
            # Only the rest of the prompt (this conversation) is new
            rest = prompt_as_text(row)[len(prefix):]
            rest_ids = phi_tokenizer(rest, return_tensors='pt', add_special_tokens=False)['input_ids'].to(device)
            logits = phi(rest_ids, past_key_values=cache).logits[0, -1].float().cpu().numpy()
            cache.crop(-rest_ids.shape[1])           # forget this conversation, keep the shared beginning
            scores.append([logits[ids].max() for ids in letter_ids])
    return np.array(scores)


def decide(scores, topics):
    # B: the best of all 11 intents; A: the best of the topic's own intents
    pred_all = np.array(ALL_INTENTS)[scores.argmax(axis=1)]
    pred_within = []
    for row_scores, topic in zip(scores, topics):
        own_columns = [ALL_INTENTS.index(name) for name in INTENTS[topic]]
        best_own = own_columns[int(np.argmax(row_scores[own_columns]))]
        pred_within.append(ALL_INTENTS[best_own])
    return np.array(pred_within), pred_all


predictions = {}
for with_context in (False, True):
    name = 'Phi-3.5 (with context)' if with_context else 'Phi-3.5 (no context)'
    predictions[name] = decide(phi_letter_scores(sample, with_context), sample.topic)

### 9.5 The comparison models

In [ ]:
convlab_tokenizer = AutoTokenizer.from_pretrained('ConvLab/t5-small-nlu-multiwoz21-context3')
convlab = AutoModelForSeq2SeqLM.from_pretrained('ConvLab/t5-small-nlu-multiwoz21-context3').to(device).eval()
ACT = re.compile(r'\[([^\]]*)\]\[([^\]]*)\]\(([^)]*)\)')          # [intent][domain]([slot][value],...)


def convlab_intent(acts_text, topic=None):
    # Translate ConvLab's dialogue acts into one of our intents
    acts = ACT.findall(acts_text)
    booking = any('[book' in slots for intent, domain, slots in acts)     # e.g. [book people][4]
    if topic is None:                                         # B: the topic must come from the acts
        domains = [domain for intent, domain, slots in acts if domain in TOPICS]
        if not domains:
            return 'no decision'
        topic = Counter(domains).most_common(1)[0][0]
    topic_intents = list(INTENTS[topic])                      # [find_..., book_...] or a single intent
    if booking and len(topic_intents) == 2:
        return topic_intents[-1]                              # book_...
    return topic_intents[0]                                   # find_...


# ConvLab reads the previous turns and the utterance, one per line
convlab_outputs = []
with torch.no_grad():
    for _, row in sample.iterrows():
        lines = [f'{speaker}: {text}' for speaker, text in row.context] + [f'user: {row.utterance}']
        inputs = convlab_tokenizer('\n'.join(lines), return_tensors='pt').to(device)
        output = convlab.generate(**inputs, max_new_tokens=100)
        convlab_outputs.append(convlab_tokenizer.decode(output[0], skip_special_tokens=True))

convlab_within = [convlab_intent(output, topic) for output, topic in zip(convlab_outputs, sample.topic)]
convlab_all = [convlab_intent(output) for output in convlab_outputs]
predictions['ConvLab T5 (MultiWOZ)'] = (np.array(convlab_within), np.array(convlab_all))

# TF-IDF classifier (Part 8): probabilities, with the columns in the order of ALL_INTENTS
columns_in_intent_order = [list(classifier.classes_).index(name) for name in ALL_INTENTS]
tfidf_scores = classifier.predict_proba(sample.utterance)[:, columns_in_intent_order]
predictions['TF-IDF + logistic regression'] = decide(tfidf_scores, sample.topic)

# Sentence-Transformer (Part 7): cosine similarity between utterances and intent descriptions
st_utterances = utterance_vectors['Sentence-Transformer'][sample.index]
st_intents = np.vstack([intent_vectors_by_model['Sentence-Transformer'][name] for name in ALL_INTENTS])
st_utterances = st_utterances / np.linalg.norm(st_utterances, axis=1, keepdims=True)
st_intents = st_intents / np.linalg.norm(st_intents, axis=1, keepdims=True)
predictions['Sentence-Transformer'] = decide(st_utterances @ st_intents.T, sample.topic)

### 9.6 Results

In [ ]:
MODELS = ['Sentence-Transformer', 'TF-IDF + logistic regression', 'ConvLab T5 (MultiWOZ)',
          'Phi-3.5 (no context)', 'Phi-3.5 (with context)']
present = sorted(set(sample.intent))                 # the intents that occur in the sample

table_A, table_B = {}, {}
for model_name in MODELS:
    pred_within, pred_all = predictions[model_name]
    table_A[model_name], table_B[model_name] = {}, {}
    for topic in FIVE_TOPICS:
        in_topic = (sample.topic == topic).to_numpy()
        topic_intents = list(INTENTS[topic])
        if len(topic_intents) == 2:
            table_A[model_name][topic] = f1_score(sample.intent[in_topic], pred_within[in_topic],
                                                  labels=topic_intents, average='macro')
        table_B[model_name][topic] = f1_score(sample.intent[in_topic], pred_all[in_topic],
                                              labels=topic_intents, average='macro', zero_division=0)
    table_B[model_name]['all intents (macro-F1)'] = f1_score(sample.intent, pred_all, labels=present, average='macro')
    table_B[model_name]['all intents (accuracy)'] = accuracy_score(sample.intent, pred_all)

print('A – find vs book within the topic (macro-F1)')
display(pd.DataFrame(table_A).round(3))
print('B – among all 11 intents')
display(pd.DataFrame(table_B).round(3))

In [ ]:
results_B = pd.DataFrame(table_B)
colors = ['#6250d6', '#2a78d6', '#9e9e9e', '#eda100', '#e34948']      # one colour per model, in MODELS order
x = np.arange(len(results_B.index))
width = 0.16

fig, ax = plt.subplots(figsize=(14, 5.5))
for k, (model_name, color) in enumerate(zip(MODELS, colors)):
    ax.bar(x + (k - 2) * width, results_B[model_name], width, color=color, label=model_name)
ax.set_xticks(x)
ax.set_xticklabels([row.capitalize() if row in FIVE_TOPICS else row for row in results_B.index], rotation=15)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Intent prediction among all 11 intents (B) on 1,000 unseen utterances', fontweight='bold')
ax.legend(ncol=3, loc='upper left', fontsize=9)
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p9_llm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
pd.crosstab(sample.intent, predictions['Phi-3.5 (with context)'][1],
            rownames=['true intent'], colnames=['Phi-3.5 answer'])

## Part 10 – My Initiative: A More Reliable Evaluation of the LLM
### 10.1 Confidence intervals and paired tests

In [ ]:
y_true = sample.intent.to_numpy()
topic_of = sample.topic.to_numpy()

# Row positions of the utterances of each dialogue in the sample
rows_of_dialogue = list(sample.reset_index(drop=True).groupby('dialogue_id').indices.values())

# 1,000 bootstrap resamples: draw dialogues with replacement and take all their utterances
rng_boot = np.random.default_rng(0)
resamples = []
for _ in range(1000):
    drawn = rng_boot.integers(len(rows_of_dialogue), size=len(rows_of_dialogue))
    resamples.append(np.concatenate([rows_of_dialogue[dialogue] for dialogue in drawn]))

present = sorted(set(y_true))


def macro_f1(rows, pred, topic=None):
    # Macro-F1 on the given rows; for one topic, or over all intents when topic is None
    if topic is not None:
        rows = rows[topic_of[rows] == topic]
        labels = list(INTENTS[topic])
    else:
        labels = present
    return f1_score(y_true[rows], pred[rows], labels=labels, average='macro', zero_division=0)


everything = np.arange(len(sample))
boot = {}                         # boot[model][topic or 'all intents'] = the 1,000 bootstrap scores
ci_table = {}
for model_name in MODELS:
    pred_all = predictions[model_name][1]
    boot[model_name], ci_table[model_name] = {}, {}
    for topic in FIVE_TOPICS + [None]:
        key = 'all intents' if topic is None else topic
        scores = np.array([macro_f1(resample, pred_all, topic) for resample in resamples])
        boot[model_name][key] = scores
        low, high = np.percentile(scores, [2.5, 97.5])
        ci_table[model_name][key] = f'{macro_f1(everything, pred_all, topic):.2f} ({low:.2f}–{high:.2f})'

print('Macro-F1 among all 11 intents (B), with 95% confidence intervals')
pd.DataFrame(ci_table)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for k, (model_name, color) in enumerate(zip(MODELS, colors)):
    scores = boot[model_name]['all intents']
    point = macro_f1(everything, predictions[model_name][1])
    low, high = np.percentile(scores, [2.5, 97.5])
    ax.barh(k, point, color=color, height=0.6)
    ax.errorbar(point, k, xerr=[[point - low], [high - point]], fmt='none', ecolor='black', capsize=4)
    ax.text(high + 0.01, k, f'{point:.2f}', va='center', fontsize=9)
ax.set_yticks(range(len(MODELS)))
ax.set_yticklabels(MODELS)
ax.set_xlim(0, 1)
ax.set_xlabel('Macro-F1 among all intents, with 95% confidence interval')
ax.set_title('How certain are the Part 9 scores?', fontweight='bold')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p10_confidence_intervals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Paired bootstrap test: Phi-3.5 (with context) against every other model, on the same resamples
reference = boot['Phi-3.5 (with context)']['all intents']
rows = []
for model_name in MODELS:
    if model_name == 'Phi-3.5 (with context)':
        continue
    difference = boot[model_name]['all intents'] - reference
    low, high = np.percentile(difference, [2.5, 97.5])
    significant = not (low <= 0 <= high)             # the 95% interval does not include 0
    rows.append({'model': model_name, 'difference to Phi-3.5 (with context)': round(difference.mean(), 3),
                 '95% interval': f'{low:+.3f} to {high:+.3f}', 'significant': significant})
pd.DataFrame(rows).set_index('model')

### 10.2 Evaluation by type of utterance

In [ ]:
# For every labelled utterance: do its own dialogue acts name a topic? (same loop and order as in Part 6)
self_contained = []
for dialogue in dialogues:
    for turn in dialogue['turns']:
        if turn['speaker'] != 'USER':
            continue
        acts = dialog_acts[dialogue['dialogue_id']][turn['turn_id']]['dialog_act']
        names_a_topic = any(act.split('-')[0].lower() in TOPICS for act in acts)
        for frame in turn['frames']:
            if frame['state']['active_intent'] != 'NONE':
                self_contained.append(names_a_topic)
labelled['self_contained'] = self_contained

groups = {'self-contained': labelled.loc[sample.index, 'self_contained'].to_numpy()}
groups['needs context'] = ~groups['self-contained']

rows = {}
for model_name in MODELS:
    pred_within, pred_all = predictions[model_name]
    rows[model_name] = {}
    for group, in_group in groups.items():
        rows[model_name][f'{group}: B accuracy'] = accuracy_score(y_true[in_group], pred_all[in_group])
        rows[model_name][f'{group}: A accuracy'] = accuracy_score(y_true[in_group], pred_within[in_group])
print({group: int(in_group.sum()) for group, in_group in groups.items()}, 'utterances')
pd.DataFrame(rows).T.round(3)

### 10.3 Prompt robustness

In [ ]:
def pick_examples(seed):
    # The same rule as in Part 9; seed 0 gives exactly the Part 9 examples
    rng = np.random.default_rng(seed)
    chosen = []
    for name in ALL_INTENTS:
        candidates = train[(train.intent == name) & (train.utterance.str.split().str.len() >= 6)]
        chosen.append(labelled.loc[candidates.index[rng.integers(len(candidates))]])
    return chosen


# Run Phi-3.5 with context again with three other sets of examples (build_prompt uses the global `examples`)
part9_examples = examples
robust = {0: predictions['Phi-3.5 (with context)']}
for seed in (1, 2, 3):
    examples = pick_examples(seed)
    robust[seed] = decide(phi_letter_scores(sample, with_context=True), sample.topic)
examples = part9_examples

# Macro-F1 of B per topic and over all intents, for every set of examples
rows = {}
for seed, (pred_within, pred_all) in robust.items():
    label = f'seed {seed} (Part 9)' if seed == 0 else f'seed {seed}'
    scores = {topic: macro_f1(everything, pred_all, topic) for topic in FIVE_TOPICS}
    scores['all intents'] = macro_f1(everything, pred_all)
    rows[label] = scores
robustness = pd.DataFrame(rows).T
robustness.loc['mean'] = robustness.mean()
robustness.loc['spread (max - min)'] = robustness.iloc[:4].max() - robustness.iloc[:4].min()
print('Phi-3.5 (with context): macro-F1 among all intents (B) with four different sets of prompt examples')
robustness.round(3)